# Exploração e coleta dos dados de votações da Câmara

Este notebook documenta a exploração inicial dos dados abertos da Câmara dos Deputados
usados no projeto O Gabinete. Aqui baixamos os arquivos brutos, inspecionamos sua estrutura
e geramos as versões limpas usadas no cálculo de similaridade entre deputados.

A lógica final de produção está nos arquivos `coleta.py` e `processamento.py`, dentro de
`pipeline/`. Este notebook serve como registro exploratório e visual do processo.


In [ ]:
import requests
import pandas as pd
import os

## Definindo o período e as fontes de dados

Como o grupo ainda não fechou oficialmente o período de análise, usamos 2024 como
referência provisória (item #2 do board).

Duas fontes são usadas:
- `votacoesVotos`: o voto individual de cada deputado em cada votação
- `votacoes`: metadados sobre cada votação (usado depois para decidir filtros)

In [ ]:
ANO_REFERENCIA = 2024

url_votos = f'https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-{ANO_REFERENCIA}.csv'
url_votacoes = f'https://dadosabertos.camara.leg.br/arquivos/votacoes/csv/votacoes-{ANO_REFERENCIA}.csv'

## Baixando os dados brutos

Verifica se os arquivos já existem em `dados/brutos/` antes de baixar, para não repetir
o download toda vez que o notebook rodar. Os caminhos são calculados a partir da raiz do
projeto (via `encontrar_raiz`), então funciona independente de onde o notebook estiver salvo.

In [16]:
from pathlib import Path

def encontrar_raiz(marcador="requirements.txt"):
    caminho = Path.cwd()
    while not (caminho / marcador).exists():
        caminho = caminho.parent
    return caminho

RAIZ = encontrar_raiz()
pasta = RAIZ / "dados" / "brutos"
pasta.mkdir(parents=True, exist_ok=True)

for url in [url_votos, url_votacoes]:
    nome_arquivo = url.split('/')[-1]
    caminho_arquivo = pasta / nome_arquivo

    if not caminho_arquivo.exists():
        print(f'Baixando {nome_arquivo}...')
        response = requests.get(url)
        with open(caminho_arquivo, 'wb') as f:
            f.write(response.content)
    else:
        print(f'{nome_arquivo} já existe. Pulando download.')

with open(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', 'r', encoding='utf-8') as f:
    print(f.readline())

Baixando votacoesVotos-2024.csv...
Baixando votacoes-2024.csv...
﻿"idVotacao";"uriVotacao";"dataHoraVoto";"voto";"deputado_id";"deputado_uri";"deputado_nome";"deputado_siglaPartido";"deputado_uriPartido";"deputado_siglaUf";"deputado_idLegislatura";"deputado_urlFoto"



### Observação sobre o formato do CSV

O cabeçalho impresso acima revela dois detalhes importantes do arquivo da Câmara:
- separador é `;`, não vírgula (por isso o `sep=';'` na leitura)
- o arquivo tem um BOM no início (por isso o `encoding='utf-8-sig'`, que remove esse
  caractere invisível automaticamente)

## Lendo o CSV de votos para o pandas

In [17]:
df = pd.read_csv(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

## Limpando o arquivo de votos

O arquivo bruto traz dados do deputado (nome, partido, UF, foto) repetidos em toda linha,
o que é redundante e deixa o arquivo maior do que precisa. Mantemos só o essencial para o
cálculo de similaridade: `idVotacao`, `deputado_id` e `voto`, além de `siglaPartido` e
`siglaUf` (usados depois para colorir os vértices do grafo).

O resultado é salvo em `dados/processado/votos-{ano}-limpo.csv`.

In [18]:
colunas_uteis = ['idVotacao', 'deputado_id', 'voto', 'deputado_siglaPartido', 'deputado_siglaUf']
df_limpo = df[colunas_uteis]

(RAIZ / 'dados' / 'processado').mkdir(parents=True, exist_ok=True)
df_limpo.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-limpo.csv', index=False)


## Gerando a tabela de deputados

Como os dados do deputado se repetem em cada linha de voto, extraímos um registro único
por `deputado_id` a partir do próprio arquivo bruto, incluindo a URL da foto (que já vem
válida no CSV, sem precisar de chamada extra à API).

Resultado salvo em `dados/processado/deputados.csv`.

In [19]:
deputados_df = df.drop_duplicates(subset='deputado_id')[
    ['deputado_id', 'deputado_nome', 'deputado_siglaPartido', 'deputado_siglaUf', 'deputado_urlFoto']
].copy()

deputados_df.to_csv(RAIZ / 'dados' / 'processado' / 'deputados.csv', index=False)

In [41]:
url_objetos = f'https://dadosabertos.camara.leg.br/arquivos/votacoesObjetos/csv/votacoesObjetos-{ANO_REFERENCIA}.csv'

caminho_objetos = pasta / f'votacoesObjetos-{ANO_REFERENCIA}.csv'

if not caminho_objetos.exists():
    print(f'Baixando votacoesObjetos-{ANO_REFERENCIA}.csv...')
    response = requests.get(url_objetos)
    with open(caminho_objetos, 'wb') as f:
        f.write(response.content)
else:
    print('Já existe. Pulando download.')

df_objetos = pd.read_csv(caminho_objetos, sep=';', encoding='utf-8-sig')

#print(df_objetos.shape)
#print(df_objetos.columns)

#df_objetos.head()

Já existe. Pulando download.


In [25]:
total_votacoes = df_objetos['idVotacao'].nunique()
votacoes_filtradas = df_objetos[df_objetos['proposicao_siglaTipo'].isin(['PL', 'PEC', 'MPV', 'PLP'])]['idVotacao'].nunique()

print(total_votacoes, votacoes_filtradas)

9541 793


In [ ]:
resposta = requests.get('https://dadosabertos.camara.leg.br/api/v2/referencias/tiposProposicao')
tipos_referencia = resposta.json()['dados']

for tipo in tipos_referencia:
    print(tipo['sigla'], '-', tipo['nome'])

In [42]:
url_afetadas = f'https://dadosabertos.camara.leg.br/arquivos/votacoesProposicoes/csv/votacoesProposicoes-{ANO_REFERENCIA}.csv'

caminho_afetadas = pasta / f'votacoesProposicoes-{ANO_REFERENCIA}.csv'

if not caminho_afetadas.exists():
    print(f'Baixando votacoesProposicoes-{ANO_REFERENCIA}.csv...')
    response = requests.get(url_afetadas)
    with open(caminho_afetadas, 'wb') as f:
        f.write(response.content)
else:
    print('Já existe. Pulando download.')

df_afetadas = pd.read_csv(caminho_afetadas, sep=';', encoding='utf-8-sig')

#print(df_afetadas.shape)
#print(df_afetadas.columns)
#df_afetadas.head()

Já existe. Pulando download.


In [33]:
df_afetadas[df_afetadas['idVotacao'] == '2355879-42']

,idVotacao,uriVotacao,data,descricao,proposicao_id,proposicao_uri,proposicao_titulo,proposicao_ementa,proposicao_codTipo,proposicao_siglaTipo,proposicao_numero,proposicao_ano
2136,2355879-42,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-02-06,Aprovado o Substitutivo ao Projeto de Lei nº 1...,2355879,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 1825/2023,Cria a Semana Cultural Interescolar nas escola...,139,PL,1825,2023.0


In [36]:
print(df_afetadas['proposicao_siglaTipo'].value_counts())

print(df_afetadas['idVotacao'].nunique())

proposicao_siglaTipo
PL     4367
REQ     754
PDL     710
TVR     487
RIC     460
PLP     306
MPV     109
PEC      83
MSC      62
PRC      40
SUG      20
PLN      16
PDC       6
REC       6
INC       5
CMC       3
SLD       3
REP       2
REL       2
RPD       1
SOR       1
Name: count, dtype: int64
7140


In [37]:
tipos_de_merito = ['PL', 'PEC', 'MPV', 'PLP', 'PDC', 'PDL', 'PLN']

votacoes_de_merito = df_afetadas[df_afetadas['proposicao_siglaTipo'].isin(tipos_de_merito)]['idVotacao'].unique()

print(len(votacoes_de_merito))

5411


In [38]:
tipos_de_merito = ['PL', 'PEC', 'MPV', 'PLP', 'PDC', 'PDL', 'PLN']

votacoes_de_merito = df_afetadas[df_afetadas['proposicao_siglaTipo'].isin(tipos_de_merito)]['idVotacao'].unique()

df_limpo_filtrado = df_limpo[df_limpo['idVotacao'].isin(votacoes_de_merito)]

print(df_limpo_filtrado.shape)
print(df_limpo_filtrado['idVotacao'].nunique())

(114631, 5)
425


In [39]:
df_limpo_filtrado.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-merito.csv', index=False)